# MNIST Data Preparation for Dropout Uncertainty Classification

This notebook prepares the MNIST dataset for the dropout uncertainty classification experiments.

In [1]:
import numpy as np
import os
from sklearn.datasets import fetch_openml
from sklearn.model_selection import KFold

In [2]:
# Create directory structure
os.makedirs("./data/MNIST/data", exist_ok=True)
os.makedirs("./data/MNIST/results", exist_ok=True)

print("Downloading MNIST dataset...")

In [3]:
# Load MNIST - this handles the download
try:
    # Method 1: Using fetch_openml
    mnist = fetch_openml('mnist_784', version=1, parser='auto')
    X = mnist.data.astype(np.float32).to_numpy()  # Convert to numpy array
    y = mnist.target.astype(np.int32).to_numpy()  # Convert to numpy array
except Exception as e:
    print(f"Error with fetch_openml: {e}")
    print("Trying alternative method...")
    try:
        # Method 2: Using fetch_mldata (older method)
        from sklearn.datasets import fetch_mldata
        mnist = fetch_mldata('MNIST original')
        X = mnist.data.astype(np.float32)
        y = mnist.target.astype(np.int32)
    except Exception as e2:
        print(f"Error with fetch_mldata: {e2}")
        print("Falling back to sklearn digits dataset (not MNIST but similar structure)...")
        # Method 3: Fall back to digits dataset
        from sklearn.datasets import load_digits
        digits = load_digits()
        X = digits.data.astype(np.float32)
        y = digits.target.astype(np.int32)
        print("Using digits dataset instead of MNIST. This has 8x8 images (64 features) instead of 28x28 (784).")

print(f"Dataset loaded. X shape: {X.shape}, y shape: {y.shape}")

Dataset loaded. X shape: (70000, 784), y shape: (70000,)


In [4]:
# Normalize data to [0, 1] range if needed
if X.max() > 1.0:
    X = X / 255.0 if X.max() > 16 else X / 16.0

# Combine features and target
data = np.column_stack((X, y))

print("Saving data files...")

Saving data files...


In [5]:
# Save main data file
np.savetxt("./data/MNIST/data/data.txt", data)

# Create index files
feature_indices = np.arange(X.shape[1])  # Adapt to the actual number of features
np.savetxt("./data/MNIST/data/index_features.txt", feature_indices, fmt='%d')
np.savetxt("./data/MNIST/data/index_target.txt", [X.shape[1]], fmt='%d')

In [6]:
# Create hyperparameter files
np.savetxt("./data/MNIST/data/n_hidden.txt", [100], fmt='%d')
np.savetxt("./data/MNIST/data/n_epochs.txt", [40], fmt='%d')
np.savetxt("./data/MNIST/data/n_splits.txt", [5], fmt='%d')  # Using 5 splits to keep runtime reasonable
np.savetxt("./data/MNIST/data/n_classes.txt", [10], fmt='%d')
np.savetxt("./data/MNIST/data/dropout_rates.txt", [0.1, 0.2, 0.5], fmt='%.2f')
np.savetxt("./data/MNIST/data/tau_values.txt", [0.01, 0.1, 1.0, 10.0], fmt='%.2f')

In [7]:
print("Creating train-test splits...")
# For efficiency, let's create a reduced dataset with 10,000 samples
# This will make the experiment run much faster for demonstration
if len(X) > 10000:
    print("Creating a smaller subset (10,000 samples) for faster processing...")
    indices = np.random.choice(X.shape[0], 10000, replace=False)
    X_reduced = X[indices]
    y_reduced = y[indices]
else:
    print("Dataset already small enough, using all samples...")
    X_reduced = X
    y_reduced = y
    indices = np.arange(len(X))

Creating train-test splits...
Creating a smaller subset (10,000 samples) for faster processing...


In [8]:
# Create train-test splits
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for i, (train_idx, test_idx) in enumerate(kf.split(X_reduced)):
    if len(X) > 10000:
        # Save indices into the original dataset
        np.savetxt(f"./data/MNIST/data/index_train_{i}.txt", indices[train_idx], fmt='%d')
        np.savetxt(f"./data/MNIST/data/index_test_{i}.txt", indices[test_idx], fmt='%d')
    else:
        # Save indices directly
        np.savetxt(f"./data/MNIST/data/index_train_{i}.txt", train_idx, fmt='%d')
        np.savetxt(f"./data/MNIST/data/index_test_{i}.txt", test_idx, fmt='%d')
    print(f"Split {i+1}/5 created. Train size: {len(train_idx)}, Test size: {len(test_idx)}")

Split 1/5 created. Train size: 8000, Test size: 2000
Split 2/5 created. Train size: 8000, Test size: 2000
Split 3/5 created. Train size: 8000, Test size: 2000
Split 4/5 created. Train size: 8000, Test size: 2000
Split 5/5 created. Train size: 8000, Test size: 2000


In [9]:
print("Dataset preparation complete!")
print(f"Data saved to ./data/MNIST/data/")
print(f"Number of features: {X.shape[1]}")

Dataset preparation complete!
Data saved to ./data/MNIST/data/
Number of features: 784
